# convtranspose-bn-activation-block — faded example 3: Complete output_padding for a 5x5 -> 11x11 upsample

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `convtranspose-bn-activation-block`. Running the beacon reports progress on the `GAN: ConvT+BN+Activation block` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: ConvT+BN+Activation block` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`convtranspose-bn-activation-block`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "convtranspose-bn-activation-block"
DD_SUBTOPIC = "GAN: ConvT+BN+Activation block"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

For odd-sized targets the strided ConvTranspose needs `output_padding` to disambiguate the inverse of a strided conv. The size formula is `H_out = (H_in-1)*stride - 2*padding + kernel + output_padding`, and the constraint `output_padding < max(stride, dilation)` must hold. The block is still `ConvT(bias=False) -> BN -> ReLU`.

## Faded exercise 3

Build a generator block that upsamples `(B, in_c, 5, 5)` to `(B, out_c, 11, 11)` using `stride=2, kernel_size=4, padding=1`. Everything is provided except the `output_padding` value. Compute the integer `output_padding` that makes the output exactly 11x11.

**Fill in:** Computes output_padding = 1, the value that makes H_out = (5-1)*2 - 2*1 + 4 + output_padding = 11.

In [ ]:
import torch.nn as nn

def build_odd_block(in_c, out_c):
    output_padding = None  # TODO: integer output_padding so (5-1)*2 - 2*1 + 4 + output_padding == 11
    return nn.Sequential(
        nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1, output_padding=output_padding, bias=False),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
    )

t.manual_seed(0)
block = build_odd_block(32, 16)
x = t.randn(2, 32, 5, 5)
out = block(x)
print(tuple(out.shape))


def _test():
    import torch.nn as nn
    t.manual_seed(0)
    block = build_odd_block(32, 16)
    assert isinstance(block[0], nn.ConvTranspose2d), 'first layer must be ConvTranspose2d'
    assert block[0].output_padding == (1, 1), 'output_padding must be 1'
    assert block[0].output_padding[0] < max(block[0].stride[0], block[0].dilation[0]), 'output_padding must be < max(stride, dilation)'
    x = t.randn(2, 32, 5, 5)
    out = block(x)
    assert tuple(out.shape) == (2, 16, 11, 11), f'expected (2,16,11,11) got {tuple(out.shape)}'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn as nn

def build_odd_block(in_c, out_c):
    output_padding = 1  # (5-1)*2 - 2*1 + 4 + 1 = 11
    return nn.Sequential(
        nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1, output_padding=output_padding, bias=False),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
    )

t.manual_seed(0)
block = build_odd_block(32, 16)
x = t.randn(2, 32, 5, 5)
out = block(x)
print(tuple(out.shape))
```
</details>